# WildLens — Custom Offline Species Classifier (Phase 2: Train)

Trains a small on-device TFLite classifier over the WildLens catalog species,
from the dataset pulled by `scripts/pull_inat_dataset.py`. See the Obsidian
runbook `custom-model-pipeline.md` for the full pipeline context.

**Run this in Google Colab with a GPU runtime** (Runtime → Change runtime type → GPU).

**Before running:**
1. Zip your dataset folder: `cd /path/to && zip -r model_training_data.zip model_training_data -x '*.csv'`
   (excluding the CSVs keeps the zip smaller — they're not needed for training)
2. Upload the zip to this Colab session (left sidebar → Files → upload), **or**
   upload it to Google Drive and mount Drive below — Drive is safer for a
   ~5 GB file since Colab session storage is wiped when the runtime recycles.

**Output of this notebook:**
- `species_id.tflite` — ~3–6 MB int8-quantized classifier
- `species_labels.json` — index → catalog species id (the model's class order)

Bring both files back to the repo at `assets/models/` for Phase 3 integration.

## 1. Setup

In [ ]:
import tensorflow as tf
print("TensorFlow:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


## 2. Get the data onto the Colab filesystem

Pick ONE of the two cells below depending on where you uploaded the zip.

In [ ]:
# Option A: uploaded model_training_data.zip directly to this Colab session
import zipfile, os

ZIP_PATH = "/content/model_training_data.zip"   # adjust if you named it differently
DATA_DIR = "/content/model_training_data"

if not os.path.exists(DATA_DIR):
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall("/content")
print("Species folders:", len(os.listdir(DATA_DIR)))


In [ ]:
# Option B: dataset zip is on Google Drive instead — mount Drive and unzip from there
# from google.colab import drive
# drive.mount('/content/drive')
# ZIP_PATH = "/content/drive/MyDrive/model_training_data.zip"  # adjust to your path
# DATA_DIR = "/content/model_training_data"
# import zipfile, os
# if not os.path.exists(DATA_DIR):
#     with zipfile.ZipFile(ZIP_PATH) as z:
#         z.extractall("/content")
# print("Species folders:", len(os.listdir(DATA_DIR)))


## 3. Load the dataset

`image_dataset_from_directory` infers one class per subfolder — since the pull
script's folders are named by catalog species id (e.g. `saguaro`, `coyote`),
**the class names ARE the catalog ids**. No taxonomy/latin-name mapping needed
(a real simplification vs. the public iNat model's `leaf_class_id` indirection
in `lib/local-identify.ts`).

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

class_names = train_ds.class_names   # index i -> catalog species id. This IS species_labels.json.
num_classes = len(class_names)
print(num_classes, "classes")
print(class_names[:10], "...")


In [ ]:
# Keep a copy of raw (unbatched, unaugmented) training images for the
# representative dataset used by INT8 quantization later.
representative_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=1,
)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)


## 4. Augmentation

Light augmentation — field photos vary in angle/lighting/zoom, but we don't
want to distort species-diagnostic features (color, shape).

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomContrast(0.15),
    tf.keras.layers.RandomBrightness(0.15),
], name="augmentation")


## 5. Build the model

Using **MobileNetV3Small** as the backbone: it's built into `tf.keras.applications`
(unlike EfficientNet-Lite, which isn't available there), is designed for mobile
inference, and quantizes to INT8 cleanly — a better fit for a hand-rolled Colab
pipeline than chasing EfficientNet-Lite conversion tooling. Either backbone
produces a plain TFLite flatbuffer that `react-native-fast-tflite` loads
identically, so this is purely a training-side convenience choice.

In [ ]:
preprocess_input = tf.keras.applications.mobilenet_v3.preprocess_input

base_model = tf.keras.applications.MobileNetV3Small(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights="imagenet",
    minimalistic=True,   # smaller variant — matters more once quantized
)
base_model.trainable = False

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.25)(x)
outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)
model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy", tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3")],
)
model.summary()


## 6. Train the head (backbone frozen)

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_accuracy", patience=4, restore_best_weights=True
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[early_stop],
)


## 7. Fine-tune

Unfreeze the top portion of the backbone and continue training at a much
lower learning rate — this recovers accuracy the frozen-backbone phase leaves
on the table, without wrecking the pretrained features.

In [ ]:
base_model.trainable = True

# Freeze everything except roughly the last third of layers.
fine_tune_at = int(len(base_model.layers) * 0.7)
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy", tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3")],
)

fine_tune_epochs = 15
history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=fine_tune_epochs,
    callbacks=[early_stop],
)


## 8. Evaluate

Look at both overall accuracy and *which* species get confused — that tells
you whether any classes need more data or should be merged/dropped for v1.

In [ ]:
val_loss, val_acc, val_top3 = model.evaluate(val_ds)
print(f"Validation top-1 accuracy: {val_acc:.1%}")
print(f"Validation top-3 accuracy: {val_top3:.1%}")


In [ ]:
# Per-class confusion — which species does the model most often mix up?
import numpy as np

y_true, y_pred = [], []
for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

y_true, y_pred = np.array(y_true), np.array(y_pred)

per_class_acc = {}
for i, name in enumerate(class_names):
    mask = y_true == i
    if mask.sum() > 0:
        per_class_acc[name] = (y_pred[mask] == i).mean()

worst = sorted(per_class_acc.items(), key=lambda kv: kv[1])[:15]
print("Lowest-accuracy classes (check these — may need more/cleaner data):")
for name, acc in worst:
    print(f"  {acc:.0%}  {name}")


## 9. Quantize + export to TFLite (INT8)

Full INT8 quantization (both weights and activations) needs a *representative
dataset* — a sample of real input images — so the converter can calibrate the
int8 ranges. Input/output tensors end up `uint8`, which matches the
`wantsUint8` branch already implemented in `lib/local-identify.ts`.

In [ ]:
def representative_data_gen():
    for images, _ in representative_ds.take(300):
        img = tf.image.resize(images, IMG_SIZE)
        yield [tf.cast(img, tf.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.uint8

tflite_model = converter.convert()

with open("species_id.tflite", "wb") as f:
    f.write(tflite_model)

import os
size_mb = os.path.getsize("species_id.tflite") / (1024 * 1024)
print(f"species_id.tflite: {size_mb:.1f} MB")


In [ ]:
import json

with open("species_labels.json", "w") as f:
    json.dump(class_names, f, indent=2)

print("Wrote species_labels.json —", len(class_names), "entries, index-aligned with the model output.")


## 10. Sanity-check the exported TFLite model

Run one real image through the quantized model directly (not the Keras
model) to catch a broken conversion before leaving Colab — e.g. wrong
input dtype, wrong preprocessing, or a scale/zero-point mismatch.

In [ ]:
interpreter = tf.lite.Interpreter(model_path="species_id.tflite")
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]
print("input:", input_details["shape"], input_details["dtype"])
print("output:", output_details["shape"], output_details["dtype"])

for images, labels in val_ds.take(1):
    sample_img = images[0].numpy().astype("uint8")
    true_label = class_names[labels[0].numpy()]
    break

interpreter.set_tensor(input_details["index"], sample_img[None, ...])
interpreter.invoke()
output = interpreter.get_tensor(output_details["index"])[0]

top3_idx = output.argsort()[-3:][::-1]
print(f"\nTrue label: {true_label}")
print("Top-3 predictions from the quantized TFLite model:")
for i in top3_idx:
    print(f"  {class_names[i]}: {output[i]}")


## 11. Download the artifacts

Download `species_id.tflite` and `species_labels.json` from the Colab file
browser (left sidebar), then bring them back to the repo:

```
nature-app/assets/models/species_id.tflite
nature-app/assets/models/species_labels.json
```

**Next: Phase 3 (app integration)** — see `custom-model-pipeline.md` Phase 3.
Since class names are already catalog ids (not scientific names), the taxonomy
mapping in `lib/local-identify.ts` gets simpler: swap the `leaf_class_id`
lookup for a direct `species_labels[i] -> catalog id` array index, and bundle
both files via `require()` instead of downloading them at runtime.

In [ ]:
from google.colab import files
files.download("species_id.tflite")
files.download("species_labels.json")
